# PharmaIntel-AI: reproducible literature-to-signal pipeline

This notebook documents the run and opens the generated evidence. It does not infer clinical causality. The full 3,915-sentence run uses the pinned BC5CDR checkpoint on CPU and may download weights from the public Hugging Face Hub on first use. Run from the repository checkout; data and model artifacts stay local.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import networkx as nx
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook inside a PharmaIntel repository checkout.')
SENTENCES = ROOT / 'data/processed/pubmed/pubmed_sentences.csv'
assert SENTENCES.exists(), 'Retrieve PubMed data first; see README.md.'
print('Project root:', ROOT)

## 1. Corpus and provenance

The retrieval CLI uses NCBI E-utilities through Biopython. It does not use an LLM API. `NCBI_EMAIL` identifies the client; `NCBI_API_KEY` is optional. The local sentence CSV retains PMID, query, and retrieval timestamp.

In [ ]:
sentences = pd.read_csv(SENTENCES, dtype={'pmid': str}, keep_default_na=False)
assert {'pmid', 'sentence_index', 'sentence', 'title'}.issubset(sentences.columns)
print(f"{len(sentences):,} sentences from {sentences.pmid.nunique():,} PMIDs")
sentences.head(3)

## 2. Run pinned biomedical NER and evidence extraction

BC5CDR labels are `Chemical` and `Disease`; the project preserves these source labels. Its `Disease` class is not ADR-specific. The case-study ranking filters to clozapine and a predeclared blood-event vocabulary. Run the script cell below to reproduce transformer inference, pair-scoped relation cues, ranking, and GraphML. CPU inference on the current snapshot took about 91 seconds after the model was cached.

In [ ]:
run = subprocess.run([sys.executable, str(ROOT / 'scripts/run_transformer_corpus_ner.py')], cwd=ROOT, check=True, text=True, capture_output=True)
print(run.stdout[-4000:])
if run.stderr:
    print(run.stderr[-1500:])

In [ ]:
summary = json.loads((ROOT / 'results/transformer_corpus_summary.json').read_text())
evidence = pd.read_csv(ROOT / 'data/processed/pubmed/transformer_case_study_evidence.csv', dtype={'pmid': str}, keep_default_na=False)
signals = pd.read_csv(ROOT / 'results/transformer_case_study_signals.csv')
print(json.dumps({k: summary[k] for k in ['sentences_scanned','unique_pmids_scanned','case_study_pair_rows','case_study_relation_status_counts','evaluation']}, indent=2))
signals

## 3. Evidence traceability

Each row below connects a model-detected chemical and disease candidate to the exact source sentence and PMID. NER scores are token-classification outputs, not probabilities that a drug caused an event.

In [ ]:
example = evidence.iloc[0]
print('PMID:', example.pmid, 'https://pubmed.ncbi.nlm.nih.gov/' + str(example.pmid) + '/')
print('Title:', example.title)
print('Pair:', example.drug_surface, '→', example.adr_surface)
print('Raw model labels:', example.drug_model_label, '/', example.adr_model_label)
print('NER scores:', example.drug_model_score, example.adr_model_score)
print('Sentence status:', example.relation_status)
print('Evidence:', example.evidence_sentence)
print('Relation cues:', example.relation_cues)
print('Context cues:', example.context_cues)

## 4. NER benchmark results

These strict exact-span metrics come from the 120-row annotated NER sample, reported separately for random and target-enriched strata. Do not pool the strata. These are mention-extraction metrics, not relation or causality metrics.

In [ ]:
ner_metrics_path = ROOT / 'results/evaluation/ner_metrics.csv'
if ner_metrics_path.exists():
    ner_metrics = pd.read_csv(ner_metrics_path)
    display(ner_metrics)
else:
    print('Benchmark artifacts are not present. Complete NER annotation, then run scripts/run_ner_benchmark.py.')

## 5. Relation evaluation status

The existing relation pilot is assistant-curated and the cue rules were refined using its errors. Treat it as development diagnostics only. A separate PMID-disjoint sample is prepared for blind human review. This notebook deliberately does not fill its labels or report held-out scores.

In [ ]:
holdout = ROOT / 'data/annotated/relation_holdout_sample.csv'
if holdout.exists():
    review = pd.read_csv(holdout, dtype={'pmid': str}, keep_default_na=False)
    print(f"{len(review)} candidates; {review.review_status.eq('complete').sum()} reviewed; {review.pmid.nunique()} unique PMIDs")
    display(review[['sampling_stratum','pmid','drug','adr','sentence','gold_status','review_status']].head())
else:
    print('Create the blind review set with scripts/prepare_relation_holdout.py')
print('After all labels are entered in dashboard/relation_annotation_app.py, run scripts/evaluate_relation_holdout.py.')

## 6. Evidence graph and interpretation

The transparent prioritization score is `log1p(unique positive PMIDs) × assertion consistency`. Co-mentions do not contribute positive support. This is a literature-triage score only; publication frequency is not incidence or risk.

In [ ]:
graph_path = ROOT / 'results/pharmaintel_transformer_case_graph.graphml'
if graph_path.exists():
    graph = nx.read_graphml(graph_path)
    print(f"Graph: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges")
    evidence_nodes = [(n,d) for n,d in graph.nodes(data=True) if d.get('node_type') == 'evidence']
    print('Evidence nodes:', len(evidence_nodes))
    if evidence_nodes:
        n, d = evidence_nodes[0]
        print('Example provenance:', d.get('pmid'), d.get('sentence'))
else:
    print('Run corpus inference first to create the evidence graph.')

## API and deployment notes

Runtime data acquisition calls NCBI PubMed E-utilities. Hugging Face is used to download the pinned public model files; inference is local PyTorch, not a hosted inference API. GitHub is only the code/data repository. The hosted Streamlit app reads precomputed evidence assets and does not invoke NCBI or Hugging Face per visitor. See `docs/deployment.md` for private hosting setup.